# __NB03: Path Generation and Recommendation__

*Author: Shilpa Kibe. Licensed under CC BY-NC 4.0.*

**Continuing from NB02:**
- NB02 computed skill gaps for 55 focus-role employees (`employee_ksb_gaps.csv`).
- NB02 matched 154 courses to 30 KSBs with TF-IDF (production) and MiniLM (comparison).
- A rule approved 11 top matches and sent 19 for L&D review (ADR-0042).
- NB03 resolves that review, then turns gaps into ordered personal learning paths.

### __Overview__

- Step 3.1: Setup and load tables
- Step 3.2: L&D review of the 19 `ld_reviewed` matches (human-in-the-loop)
- Step 3.3: Join approved matches to open gaps
- Step 3.4: Sequence each path and save `learning_path_recommendations.csv`
- Step 3.5: Per-employee access demo (privacy)
- Step 3.6: KPI Impact Rollup
- Step 3.7: Summary and handoff to NB04

### __Step 3.1: Setup and load tables__

- Mount Drive safely (ADR-0033).
- Load the input and NB02 output tables used in this notebook.

In [1]:
# Step 3.1: Mount Drive (idempotent, ADR-0033), import libraries, load tables

import os
import pandas as pd
from google.colab import drive

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive already mounted, skipping remount, nothing deleted.")

ROOT_DIR = '/content/drive/MyDrive/Portfolio/LuminaLearningGroup'
RAW_DIR = os.path.join(ROOT_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT_DIR, 'data', 'processed')
OUTPUT_DIR = os.path.join(ROOT_DIR, 'outputs')
RANDOM_STATE = 42

# Step 3.1: Read CSVs from data/raw and data/processed
courses = pd.read_csv(os.path.join(RAW_DIR, 'courses.csv'))
ksbs = pd.read_csv(os.path.join(RAW_DIR, 'ksbs.csv'))
competencies = pd.read_csv(os.path.join(RAW_DIR, 'competencies.csv'))
employees = pd.read_csv(os.path.join(RAW_DIR, 'employees.csv'))
employees_pii = pd.read_csv(os.path.join(RAW_DIR, 'employees_pii.csv'))
kpis = pd.read_csv(os.path.join(RAW_DIR, 'kpis.csv'))
kpi_competency_mapping = pd.read_csv(os.path.join(RAW_DIR, 'kpi_competency_mapping.csv'))
course_ksb_mapping = pd.read_csv(os.path.join(PROCESSED_DIR, 'course_ksb_mapping.csv'))
employee_ksb_gaps = pd.read_csv(os.path.join(PROCESSED_DIR, 'employee_ksb_gaps.csv'))

print("Step 3.1 executed: tables loaded")
for name, df in [('courses.csv', courses), ('ksbs.csv', ksbs), ('competencies.csv', competencies),
                 ('employees.csv', employees), ('employees_pii.csv', employees_pii), ('kpis.csv', kpis),
                 ('kpi_competency_mapping.csv', kpi_competency_mapping),
                 ('course_ksb_mapping.csv', course_ksb_mapping), ('employee_ksb_gaps.csv', employee_ksb_gaps)]:
    print(f"{name}: {len(df)} rows, {len(df.columns)} columns")
print()
print(course_ksb_mapping[course_ksb_mapping['match_method'] == 'tfidf']['curation_status'].value_counts())

Mounted at /content/drive
Step 3.1 executed: tables loaded
courses.csv: 154 rows, 16 columns
ksbs.csv: 30 rows, 4 columns
competencies.csv: 10 rows, 2 columns
employees.csv: 430 rows, 6 columns
employees_pii.csv: 430 rows, 5 columns
kpis.csv: 4 rows, 6 columns
kpi_competency_mapping.csv: 5 rows, 3 columns
course_ksb_mapping.csv: 9240 rows, 9 columns
employee_ksb_gaps.csv: 1650 rows, 6 columns

curation_status
algo_suggested    4590
ld_reviewed         19
ld_approved         11
Name: count, dtype: int64


### __Step 3.1 Notes/Insights__

- All 9 tables loaded from Drive.
- `kpis.csv` now has 6 columns, including `kpi_type`. This came from a source fix in NB01, logged under ADR-0037.
- 55 focus-role employees x 30 KSBs = 1,650 gap rows, as built in NB02.
- TF-IDF (production) rows: 4,620. Only 30 have a curation decision; the rest stay `algo_suggested`.
- 11 matches are already approved; 19 wait for L&D review in Step 3.2.

In [2]:

# Step 3.1 check: kpis.csv columns and K2.2 wording
print(kpis.columns.tolist())
print(kpis[['kpi_id', 'kpi_text', 'target_value']])

['kpi_id', 'goal_id', 'kpi_text', 'target_value', 'target_date', 'kpi_type']
  kpi_id                                           kpi_text target_value
0   K1.1  % of client learning programs using AI-persona...          25%
1   K1.2  Average client-reported learner satisfaction s...        4.2/5
2   K2.1  % of Content Dev staff certified in at least D...          60%
3   K2.2  Average KSB gap-closure rate per employee per ...  6 KSBs/year


### __Step 3.2: L&D review of the 19 `ld_reviewed` matches__

- NB02 found two valid candidate courses for these 19 KSBs; the two matching methods disagreed or were not confident.
- In real organisations, an L&D curator reviews such cases in a review queue or an exported sheet.
- Step 3.2.1 exports a side-by-side review sheet.
- The reviewer sees both course options and their content. The answer key (`target_ksb_id`) is hidden, as in a blind review (ADR-0038).
- Step 3.2.2 loads the completed sheet back and updates `curation_status`.

In [3]:
# Step 3.2.1: Build side-by-side review sheet for ld_reviewed matches
# Reads: course_ksb_mapping.csv, courses.csv, ksbs.csv
# Saves: outputs/curation_review_sheet.csv

# Suggested = TF-IDF top course (production); Alternative = MiniLM top course (comparison)
suggested = course_ksb_mapping[(course_ksb_mapping['match_method'] == 'tfidf') &
                               (course_ksb_mapping['curation_status'] == 'ld_reviewed')]
alternative = course_ksb_mapping[(course_ksb_mapping['match_method'] == 'embedding') &
                                 (course_ksb_mapping['is_recommended'] == True)]

sheet = suggested[['ksb_id', 'course_id', 'match_score', 'match_explanation']].rename(columns={
    'course_id': 'suggested_course_id', 'match_score': 'suggested_score',
    'match_explanation': 'suggested_shared_keywords'})
sheet = sheet.merge(alternative[['ksb_id', 'course_id', 'match_score']].rename(columns={
    'course_id': 'alternative_course_id', 'match_score': 'alternative_score'}), on='ksb_id', how='left')
sheet = sheet.merge(ksbs[['ksb_id', 'competency_id', 'ksb_type', 'ksb_text']], on='ksb_id', how='left')

# Course details the reviewer needs; target labels deliberately excluded (blind review, ADR-0038)
course_cols = ['course_id', 'course_title', 'description_raw', 'course_source', 'format',
               'duration_minutes', 'ksb_layer', 'duplicate_cluster_id']
for prefix in ['suggested_', 'alternative_']:
    sheet = sheet.merge(courses[course_cols].add_prefix(prefix), on=f'{prefix}course_id', how='left')

sheet['same_course'] = sheet['suggested_course_id'] == sheet['alternative_course_id']
sheet['decision'] = ''                 # approve / switch / reject
sheet['replacement_course_id'] = ''    # only if reject and a different course should be used
sheet['reviewer_note'] = ''            # one short reason

sheet = sheet[['ksb_id', 'competency_id', 'ksb_type', 'ksb_text', 'same_course',
               'suggested_course_id', 'suggested_course_title', 'suggested_description_raw',
               'suggested_course_source', 'suggested_format', 'suggested_duration_minutes',
               'suggested_ksb_layer', 'suggested_duplicate_cluster_id', 'suggested_score',
               'suggested_shared_keywords',
               'alternative_course_id', 'alternative_course_title', 'alternative_description_raw',
               'alternative_course_source', 'alternative_format', 'alternative_duration_minutes',
               'alternative_ksb_layer', 'alternative_duplicate_cluster_id', 'alternative_score',
               'decision', 'replacement_course_id', 'reviewer_note']].sort_values('ksb_id').reset_index(drop=True)

review_path = os.path.join(OUTPUT_DIR, 'curation_review_sheet.csv')
# Guard: only export when there are matches left to review (protects the original audit copy on re-runs)
if len(sheet) > 0:
    sheet.to_csv(review_path, index=False, encoding='utf-8-sig')  # utf-8-sig opens cleanly in Excel
else:
    print("Step 3.2.1: no ld_reviewed rows left, review sheet not overwritten")

print("Step 3.2.1 executed: curation_review_sheet.csv saved to outputs")
print(f"Rows to review: {len(sheet)}")
print(f"Same course in both options: {sheet['same_course'].sum()}")
print(f"Different courses: {(~sheet['same_course']).sum()}")

Step 3.2.1 executed: curation_review_sheet.csv saved to outputs
Rows to review: 19
Same course in both options: 6
Different courses: 13


In [4]:
# Step 3.2.1a: Display the review sheet as a table (reads in-memory sheet from Step 3.2.1)

from IPython.display import display

view = sheet[['ksb_id', 'ksb_type', 'ksb_text', 'same_course',
              'suggested_course_id', 'suggested_course_title', 'suggested_ksb_layer', 'suggested_score',
              'alternative_course_id', 'alternative_course_title', 'alternative_ksb_layer', 'alternative_score']].copy()

view = view.rename(columns={
    'ksb_id': 'KSB', 'ksb_type': 'Type', 'ksb_text': 'KSB text', 'same_course': 'Same?',
    'suggested_course_id': 'Sugg ID', 'suggested_course_title': 'Suggested course',
    'suggested_ksb_layer': 'Sugg layer', 'suggested_score': 'Sugg score',
    'alternative_course_id': 'Alt ID', 'alternative_course_title': 'Alternative course',
    'alternative_ksb_layer': 'Alt layer', 'alternative_score': 'Alt score'})

print("Step 3.2.1a executed: review sheet displayed (full text in outputs/curation_review_sheet.csv)")
display(view.style
        .format({'Sugg score': '{:.3f}', 'Alt score': '{:.3f}'})
        .set_properties(**{'white-space': 'normal', 'text-align': 'left', 'vertical-align': 'top'})
        .set_properties(subset=['KSB text', 'Suggested course', 'Alternative course'], **{'min-width': '220px'})
        .hide(axis='index'))

Step 3.2.1a executed: review sheet displayed (full text in outputs/curation_review_sheet.csv)


KSB,Type,KSB text,Same?,Sugg ID,Suggested course,Sugg layer,Sugg score,Alt ID,Alternative course,Alt layer,Alt score
KSB001,knowledge,Understands capabilities/limitations of GenAI tools for content drafting,False,CRS001,Mastering AI-Augmented Content Design,skill,0.650,CRS004,Introduction to AI-Augmented Content Design,skill,0.817
KSB003,behavior,Iteratively reviews and refines AI-drafted content rather than accepting output as-is,False,CRS010,Mastering AI-Augmented Content Design,skill,0.542,CRS011,Introduction to AI-Augmented Content Design,skill,0.797
KSB005,skill,Can build/read dashboards to track learning program effectiveness,False,CRS020,Data-Driven Learning Analytics for L&D Professionals,skill,0.754,CRS019,Applying Data-Driven Learning Analytics to Real Projects,skill,0.777
KSB006,behavior,"Uses data to justify design decisions, not just intuition",False,CRS026,Mastering Data-Driven Learning Analytics,skill,0.604,CRS028,Applying Data-Driven Learning Analytics to Real Projects,skill,0.510
KSB010,knowledge,Understands principles of adaptive learning path design,False,CRS042,Personalization & Adaptive Design Fundamentals,skill,0.607,CRS041,Introduction to Personalization & Adaptive Design,skill,0.751
KSB012,behavior,"Designs for diverse learner starting points, not a single ""average"" learner",False,CRS051,Personalization & Adaptive Design for L&D Professionals,skill,0.727,CRS052,Personalization & Adaptive Design in Practice,skill,0.635
KSB013,knowledge,Knows ADDIE / competency-based design models,True,CRS055,Introduction to Instructional Design Fundamentals,skill,0.542,CRS055,Introduction to Instructional Design Fundamentals,skill,0.720
KSB014,skill,Can write measurable learning objectives,False,CRS059,Instructional Design Fundamentals for L&D Professionals,skill,0.660,CRS062,Instructional Design Fundamentals: Practical Application,skill,0.657
KSB015,behavior,Applies design consistently across content types,False,CRS064,Mastering Instructional Design Fundamentals,skill,0.462,CRS063,Applying Instructional Design Fundamentals to Real Projects,skill,0.405
KSB017,skill,Can translate vague client asks into a structured learning solution brief,True,CRS073,Stakeholder & Client Consulting Fundamentals,skill,0.791,CRS073,Stakeholder & Client Consulting Fundamentals,skill,0.661


### __Step 3.2.1 Notes/Insights__

- 19 matches exported to `outputs/curation_review_sheet.csv` for L&D review.
- 6 rows: both methods picked the same course (KSB013, 017, 019, 021, 025, 030).
  - They were flagged only because the scores differed by more than 0.10.
  - Decision: approve or reject.
- 13 rows: the methods picked different courses.
  - Decision: approve (keep suggested), switch (use alternative) or reject.
- Options often differ mainly in title wording ("Mastering" vs "Introduction to"). Format, duration and description help decide.
- Fit check: 5 of 6 knowledge KSBs have only skill-layer courses on offer, and no course is tagged behavior.
  - This is expected: ksb_layer is used as a fit check, and mismatches are noted in the rationale.

### __Step 3.2.2: Load L&D decisions and update curation status__

- Read the completed review sheet back into the pipeline.
- approve: the suggested course becomes `ld_approved`.
- switch, or reject with a replacement: the chosen course becomes `ld_modified`, and the algorithm's original pick becomes `ld_rejected`.
- reject with no replacement: `ld_rejected`, logged as a content gap.
- The completed sheet stays in `outputs/` as the audit trail, with reasons.

In [5]:
# Step 3.2.2: Load L&D decisions and update curation_status
# Reads: outputs/curation_review_sheet_completed.csv
# Saves: data/processed/course_ksb_mapping.csv (curation_status updated)

completed = pd.read_csv(os.path.join(OUTPUT_DIR, 'curation_review_sheet_completed.csv'), encoding='utf-8-sig')
completed['decision_key'] = completed['decision'].str.strip().str.split().str[0].str.lower()
completed['replacement_course_id'] = completed['replacement_course_id'].fillna('').astype(str).str.strip()

# Step 3.2.2: Validation checks before anything is written
assert len(completed) == 19, "Expected 19 reviewed rows"
assert completed['decision_key'].isin(['approve', 'switch', 'reject']).all(), "Unknown decision value"
bad_ids = set(completed.loc[completed['replacement_course_id'] != '', 'replacement_course_id']) - set(courses['course_id'])
assert not bad_ids, f"Unknown replacement course IDs: {bad_ids}"

# Step 3.2.2: Decide the final course and status per KSB
def final_choice(r):
    if r.decision_key == 'approve':
        return r.suggested_course_id, 'ld_approved'
    if r.decision_key == 'switch':
        return r.alternative_course_id, 'ld_modified'
    if r.replacement_course_id != '':
        return r.replacement_course_id, 'ld_modified'
    return None, 'ld_rejected'   # content gap

completed[['final_course_id', 'final_status']] = completed.apply(lambda r: pd.Series(final_choice(r)), axis=1)

# Step 3.2.2: Apply decisions to the TF-IDF (production) rows
is_tfidf = course_ksb_mapping['match_method'] == 'tfidf'
for r in completed.itertuples():
    on_ksb = is_tfidf & (course_ksb_mapping['ksb_id'] == r.ksb_id)
    sugg_row = on_ksb & (course_ksb_mapping['course_id'] == r.suggested_course_id)
    if r.final_status == 'ld_approved':
        course_ksb_mapping.loc[sugg_row, 'curation_status'] = 'ld_approved'
    else:
        course_ksb_mapping.loc[sugg_row, 'curation_status'] = 'ld_rejected'   # algorithm's pick not kept
        if pd.notna(r.final_course_id):
            new_row = on_ksb & (course_ksb_mapping['course_id'] == r.final_course_id)
            course_ksb_mapping.loc[new_row, 'curation_status'] = 'ld_modified'

# Step 3.2.2: Post-checks: one eligible course per KSB, no pending reviews
eligible = course_ksb_mapping[is_tfidf & course_ksb_mapping['curation_status'].isin(['ld_approved', 'ld_modified'])]
per_ksb = eligible.groupby('ksb_id').size()
content_gaps = sorted(set(ksbs['ksb_id']) - set(per_ksb.index))
assert (per_ksb == 1).all(), "A KSB has more than one eligible course"
assert (course_ksb_mapping['curation_status'] == 'ld_reviewed').sum() == 0, "Unresolved ld_reviewed rows remain"

course_ksb_mapping.to_csv(os.path.join(PROCESSED_DIR, 'course_ksb_mapping.csv'), index=False)

print("Step 3.2.2 executed: course_ksb_mapping.csv saved to data/processed")
print(course_ksb_mapping[is_tfidf]['curation_status'].value_counts())
print()
print(f"KSBs with an eligible course: {len(per_ksb)} of {len(ksbs)}")
print(f"Content gaps (no approved course): {content_gaps}")
print()
print(completed[['ksb_id', 'decision_key', 'suggested_course_id', 'final_course_id', 'final_status']].to_string(index=False))

Step 3.2.2 executed: course_ksb_mapping.csv saved to data/processed
curation_status
algo_suggested    4584
ld_approved         23
ld_rejected          7
ld_modified          6
Name: count, dtype: int64

KSBs with an eligible course: 29 of 30
Content gaps (no approved course): ['KSB018']

ksb_id decision_key suggested_course_id final_course_id final_status
KSB001      approve              CRS001          CRS001  ld_approved
KSB003      approve              CRS010          CRS010  ld_approved
KSB005      approve              CRS020          CRS020  ld_approved
KSB006      approve              CRS026          CRS026  ld_approved
KSB010       switch              CRS042          CRS041  ld_modified
KSB012      approve              CRS051          CRS051  ld_approved
KSB013      approve              CRS055          CRS055  ld_approved
KSB014       switch              CRS059          CRS062  ld_modified
KSB015       switch              CRS064          CRS063  ld_modified
KSB017      approve  

### __Step 3.2.2 Notes/Insights__

- All 19 reviews resolved; no `ld_reviewed` rows remain.
- Final TF-IDF statuses: 23 approved, 6 modified, 7 rejected. The 7 rejected are 6 algorithm picks replaced by the reviewer, plus KSB018.
- The reviewer changed 6 of 19 picks, mostly to favour practice and engagement over passive formats. This is the L&D judgment the algorithm cannot supply.
- Two courses now serve two KSBs each: CRS085 (KSB019, KSB020) and CRS114 (KSB025, KSB027). Each will appear once per path, with a combined note.
- 29 of 30 KSBs have one approved course. KSB018 is a content gap: it needs a workshop or simulation (build or buy).

### __Step 3.3: Join approved courses to open gaps__

- Keep only open gaps (`gap_size > 0`); zero or negative means no learning need.
- Attach the one approved course per KSB (`ld_approved` / `ld_modified`, TF-IDF rows).
- Gaps with no approved course are flagged as content gaps (KSB018).
- Output stays in memory as `path_candidates`; Step 3.4 orders and saves it.

In [6]:
# Step 3.3: Join approved courses to open gaps
# Reads (in memory): employee_ksb_gaps, course_ksb_mapping, ksbs

# Step 3.3: Open gaps only (positive gap = below requirement)
open_gaps = employee_ksb_gaps[employee_ksb_gaps['gap_size'] > 0].copy()

# Step 3.3: One approved course per KSB (production TF-IDF rows after L&D review)
approved = course_ksb_mapping[(course_ksb_mapping['match_method'] == 'tfidf') &
                              (course_ksb_mapping['curation_status'].isin(['ld_approved', 'ld_modified']))]
approved = approved[['ksb_id', 'course_id', 'match_score', 'curation_status']]

# Step 3.3: Left join keeps gaps with no approved course (content gaps)
path_candidates = (open_gaps
                   .merge(ksbs[['ksb_id', 'competency_id', 'ksb_type']], on='ksb_id', how='left')
                   .merge(approved, on='ksb_id', how='left'))
path_candidates['is_content_gap'] = path_candidates['course_id'].isna()

print("Step 3.3 executed: path_candidates built (not saved yet)")
print(f"Total gap rows: {len(employee_ksb_gaps)}")
print(f"Open gaps (gap_size > 0): {len(open_gaps)}")
print(f"Employees with at least one open gap: {open_gaps['employee_id'].nunique()} of {employee_ksb_gaps['employee_id'].nunique()}")
print(f"Open gaps with an approved course: {(~path_candidates['is_content_gap']).sum()}")
print(f"Open gaps that are content gaps: {path_candidates['is_content_gap'].sum()} "
      f"(KSBs: {sorted(path_candidates.loc[path_candidates['is_content_gap'], 'ksb_id'].unique())})")
print()
print("Open gaps per employee (min / mean / max):")
print(path_candidates.groupby('employee_id').size().agg(['min', 'mean', 'max']).round(1))
print()
print("Open gaps by gap_size:")
print(path_candidates['gap_size'].value_counts().sort_index())

Step 3.3 executed: path_candidates built (not saved yet)
Total gap rows: 1650
Open gaps (gap_size > 0): 687
Employees with at least one open gap: 55 of 55
Open gaps with an approved course: 668
Open gaps that are content gaps: 19 (KSBs: ['KSB018'])

Open gaps per employee (min / mean / max):
min      6.0
mean    12.5
max     21.0
dtype: float64

Open gaps by gap_size:
gap_size
1    540
2    144
3      3
Name: count, dtype: int64


### __Step 3.3 Notes/Insights__

- 687 of 1,650 gap rows (42%) are open gaps; the rest meet or exceed the requirement.
- All 55 focus-role employees have at least one open gap, so every one gets a path.
- 668 open gaps have an approved course. 19 are content gaps, all KSB018, so 19 employees will see the "coming soon" message.
- Most gaps are small (540 are 1 level, 144 are 2 levels). 3 rows are 3 levels: one competency (3 KSBs) for one employee. These come first in that path.
- Employees have 6 to 21 open gaps (mean 12.5), against the K2.2 target of 6 KSBs/year. Longer paths may need phasing; this is checked in the KPI rollup.

#### __Self Learning__
- We calculated gaps in NB02 (step 2.1) this way: gap_size = required_level − actual_level
- We have the proficiency scale (1 to 4). The learner rated themself.
- So if their self rating is one level below the required level it becomes a level 1 gap.

**Thus:**
Gap 3: three levels below (3 rows). These come first.
- Gap 1: one level below the requirement, for example at Practitioner when Advanced is needed (540 rows).
- Gap 2: two levels below, for example at Foundational when Advanced is needed (144 rows). These are the biggest needs, so they come first in the path.
- Gap 0 or negative: meets or exceeds the requirement, so no learning need. These were filtered out in Step 3.3.

### __Step 3.4: Sequence each path and save `learning_path_recommendations.csv`__

**Ordering rules:**
- Courses come first; content gaps ("coming soon") come last.
- Biggest gap first (gap 3, then 2, then 1).
- Within a competency: Knowledge, then Skill, then Behavior (`ksb_type`).
- Duplicate check: if two courses in one path are near-copies (same `duplicate_cluster_id`), keep the earlier one.
- Shared course: if one course covers two KSBs, it takes one place in the path and the rationale names both.
- One row per open gap is kept, so every gap stays traceable (`gap_id`) for the KPI rollup.
- `ksb_layer` is a fit check only; mismatches are noted in the rationale.

In [7]:
# Step 3.4: Sequence each employee's path, write rationale, save learning_path_recommendations.csv
# Reads (in memory): path_candidates, courses, ksbs, competencies
# Saves: data/processed/learning_path_recommendations.csv

# Step 3.4: Editable message for content gaps (L&D decision, Step 3.2)
CONTENT_GAP_MESSAGES = {
    "KSB018": "New workshop/simulation planned for Q1 2027. Meanwhile, use your manager 1:1s "
              "or the mentor-mentee programme to build this skill."
}
KSB_TYPE_ORDER = {'knowledge': 1, 'skill': 2, 'behavior': 3}

paths = (path_candidates
         .merge(competencies, on='competency_id', how='left')
         .merge(courses[['course_id', 'duplicate_cluster_id']], on='course_id', how='left'))
paths['type_order'] = paths['ksb_type'].map(KSB_TYPE_ORDER)

# Step 3.4: Order rows: courses first, biggest gap, competency, then K > S > B
paths = paths.sort_values(['employee_id', 'is_content_gap', 'gap_size', 'competency_id', 'type_order'],
                          ascending=[True, True, False, True, True]).reset_index(drop=True)

# Step 3.4: Duplicate check: a later near-copy in the same path is replaced by the earlier course
paths['duplicate_swapped_from'] = pd.NA
in_cluster = paths['duplicate_cluster_id'].notna()
first_course = paths[in_cluster].groupby(['employee_id', 'duplicate_cluster_id'])['course_id'].transform('first')
swap_idx = first_course[first_course != paths.loc[in_cluster, 'course_id']].index
paths.loc[swap_idx, 'duplicate_swapped_from'] = paths.loc[swap_idx, 'course_id']
paths.loc[swap_idx, 'course_id'] = first_course[swap_idx]

# Step 3.4: Course details for the rationale (after any swap)
paths = paths.merge(courses[['course_id', 'course_title', 'ksb_layer']], on='course_id', how='left')

# Step 3.4: Shared course: list every KSB a course covers in that employee's path
has_course = paths['course_id'].notna()
paths['covers'] = ''
paths.loc[has_course, 'covers'] = (paths[has_course].groupby(['employee_id', 'course_id'])['ksb_id']
                                   .transform(lambda s: ', '.join(s)))

# Step 3.4: sequence_order: one number per distinct course; each content gap gets its own number at the end
paths['seq_key'] = paths['course_id'].fillna('GAP_' + paths['ksb_id'])
paths['sequence_order'] = paths.groupby('employee_id')['seq_key'].transform(lambda s: pd.factorize(s)[0] + 1)

# Step 3.4: Rule-based rationale (P1 swap point; P2 = natural-language narrative)
def build_rationale(r):
    if r.is_content_gap:
        return CONTENT_GAP_MESSAGES.get(r.ksb_id, "No approved course yet. L&D has logged this as a content gap.")
    text = f"Closes a {int(r.gap_size)}-level gap in {r.competency_name} ({r.ksb_type}, {r.ksb_id})."
    others = [k for k in r.covers.split(', ') if k != r.ksb_id]
    if others:
        text += f" Also covers {', '.join(others)}."
    if r.ksb_type == 'knowledge' and r.ksb_layer == 'skill':
        text += " Practice-focused course for a knowledge need."
    elif r.ksb_type == 'skill' and r.ksb_layer == 'knowledge':
        text += " Theory-focused course; add hands-on practice."
    elif r.ksb_type == 'behavior':
        text += " Behavior need: apply it on the job after the course."
    if pd.notna(r.duplicate_swapped_from):
        text += f" Replaces near-duplicate {r.duplicate_swapped_from}."
    return text

paths['rationale'] = paths.apply(build_rationale, axis=1)
paths['recommendation_method'] = 'rule_based'
# Step 3.4: Final order: by employee, then path step (rows sharing a course sit together)
paths = paths.sort_values(['employee_id', 'sequence_order', 'type_order']).reset_index(drop=True)
paths['recommendation_id'] = [f"REC{i+1:05d}" for i in range(len(paths))]

learning_path_recommendations = paths[['recommendation_id', 'employee_id', 'course_id', 'gap_id',
                                       'sequence_order', 'recommendation_method', 'rationale']]

# Step 3.4: Checks: one row per open gap, no gap repeated
assert len(learning_path_recommendations) == len(path_candidates), "Row count must equal open gaps"
assert learning_path_recommendations['gap_id'].is_unique, "A gap appears twice"

learning_path_recommendations.to_csv(os.path.join(PROCESSED_DIR, 'learning_path_recommendations.csv'), index=False)

steps_per_emp = paths.groupby('employee_id')['sequence_order'].max()
print("Step 3.4 executed: learning_path_recommendations.csv saved to data/processed")
print(f"Rows (one per open gap): {len(learning_path_recommendations)}")
print(f"Employees with a path: {learning_path_recommendations['employee_id'].nunique()}")
print(f"Path length in steps (min / mean / max): {steps_per_emp.min()} / {steps_per_emp.mean():.1f} / {steps_per_emp.max()}")
print(f"Near-duplicate swaps applied: {len(swap_idx)}")
print(f"Rows where a shared course covers 2+ KSBs: {(paths['covers'].str.count(',') > 0).sum()}")
print(f"Content-gap rows: {paths['is_content_gap'].sum()}")
print()
sample_emp = paths['employee_id'].iloc[0]
print(f"Sample path: {sample_emp}")
print(paths[paths['employee_id'] == sample_emp][['sequence_order', 'ksb_id', 'gap_size', 'course_id', 'rationale']]
      .to_string(index=False))

Step 3.4 executed: learning_path_recommendations.csv saved to data/processed
Rows (one per open gap): 687
Employees with a path: 55
Path length in steps (min / mean / max): 5 / 11.6 / 21
Near-duplicate swaps applied: 0
Rows where a shared course covers 2+ KSBs: 98
Content-gap rows: 19

Sample path: EMP0020
 sequence_order ksb_id  gap_size course_id                                                                                                                                                rationale
              1 KSB016         1    CRS067                                                                             Closes a 1-level gap in Stakeholder & Client Consulting (knowledge, KSB016).
              2 KSB017         1    CRS073                                                                                 Closes a 1-level gap in Stakeholder & Client Consulting (skill, KSB017).
              3 KSB025         1    CRS114       Closes a 1-level gap in Competency & Skills Taxonomy De

### __Step 3.4 Notes/Insights__

- 687 recommendation rows saved, one per open gap, for all 55 employees; every gap stays traceable through `gap_id`.
- Paths run 5 to 21 steps (mean 11.6). Steps are fewer than gaps because a shared course covers two KSBs in one step.
- 98 rows use a shared course (CRS085 or CRS114), each shown once per path with "Also covers" in the rationale.
- Near-duplicate swaps: 0. No path contains two near-copy courses.
- KSB018 sits at the end of 19 paths as a "coming soon" step, with the manager 1:1 and mentoring advice.


### __Step 3.5: Per-employee access demo (privacy by design)__

- Simulates an employee logging in and viewing only their own path (ADR-0016).
- Names come from `employees_pii` only here, at the display step; no engine reads PII.
- Requests to view someone else's path are refused.
- One row per path step: a shared course shows once; content gaps show as "Coming soon".

In [8]:
# Step 3.5: Per-employee access demo (privacy by design, ADR-0016)
# Reads (in memory): learning_path_recommendations, employee_ksb_gaps, employees_pii, courses, ksbs, competencies

from IPython.display import display

def view_my_path(logged_in_id, requested_id):
    """Show a learning path only to the employee it belongs to."""
    if logged_in_id != requested_id:
        print(f"Access denied: {logged_in_id} can only view their own learning path.")
        return
    if requested_id not in set(learning_path_recommendations['employee_id']):
        print(f"No learning path found for {requested_id}.")
        return

    # PII joined only here, at the display layer
    first_name = employees_pii.loc[employees_pii['employee_id'] == requested_id, 'first_name'].iloc[0]

    my = (learning_path_recommendations[learning_path_recommendations['employee_id'] == requested_id]
          .merge(employee_ksb_gaps[['gap_id', 'ksb_id']], on='gap_id', how='left')
          .merge(ksbs[['ksb_id', 'competency_id']], on='ksb_id', how='left')
          .merge(competencies, on='competency_id', how='left')
          .merge(courses[['course_id', 'course_title', 'format', 'duration_minutes']], on='course_id', how='left'))

    # One row per step: a shared course covers several KSBs in one step
    steps = (my.groupby('sequence_order')
               .agg(Course=('course_title', 'first'), Format=('format', 'first'),
                    Minutes=('duration_minutes', 'first'), Competency=('competency_name', 'first'),
                    KSBs=('ksb_id', ', '.join), Why=('rationale', 'first'))
               .reset_index().rename(columns={'sequence_order': 'Step'}))
    steps['Course'] = steps['Course'].fillna('Coming soon (content in development)')
    steps['Format'] = steps['Format'].fillna('-')

    total_hours = steps['Minutes'].sum() / 60
    steps['Minutes'] = steps['Minutes'].apply(lambda m: '-' if pd.isna(m) else int(m))

    print(f"Hi {first_name}, here is your learning path: {len(steps)} steps, about {total_hours:.1f} hours of learning.")
    display(steps.style.set_properties(**{'white-space': 'normal', 'text-align': 'left', 'vertical-align': 'top'})
                       .hide(axis='index'))

DEMO_EMPLOYEE = 'EMP0020'

print("Step 3.5 executed: access demo")
print("--- Demo 1: employee views their own path ---")
view_my_path(DEMO_EMPLOYEE, DEMO_EMPLOYEE)
print()
print("--- Demo 2: same employee tries to view a colleague's path ---")
view_my_path(DEMO_EMPLOYEE, 'EMP0003')

Step 3.5 executed: access demo
--- Demo 1: employee views their own path ---
Hi Noah, here is your learning path: 5 steps, about 1.8 hours of learning.


Step,Course,Format,Minutes,Competency,KSBs,Why
1,Mastering Stakeholder & Client Consulting,video,19,Stakeholder & Client Consulting,KSB016,"Closes a 1-level gap in Stakeholder & Client Consulting (knowledge, KSB016)."
2,Stakeholder & Client Consulting Fundamentals,job-aid,13,Stakeholder & Client Consulting,KSB017,"Closes a 1-level gap in Stakeholder & Client Consulting (skill, KSB017)."
3,Competency & Skills Taxonomy Design Fundamentals,course,69,Competency & Skills Taxonomy Design,"KSB025, KSB027","Closes a 1-level gap in Competency & Skills Taxonomy Design (knowledge, KSB025). Also covers KSB027. Practice-focused course for a knowledge need."
4,Competency & Skills Taxonomy Design in Practice,checklist,8,Competency & Skills Taxonomy Design,KSB026,"Closes a 1-level gap in Competency & Skills Taxonomy Design (skill, KSB026)."
5,Coming soon (content in development),-,-,Stakeholder & Client Consulting,KSB018,"New workshop/simulation planned for Q1 2027. Meanwhile, use your manager 1:1s or the mentor-mentee programme to build this skill."



--- Demo 2: same employee tries to view a colleague's path ---
Access denied: EMP0020 can only view their own learning path.


### __Privacy by design (GDPR-aligned)__

Built in this pipeline (ADR-0016):
- **PII separation:** names and emails live only in `employees_pii`; engines work on `employee_id` alone (data minimisation, purpose limitation).
- **Own-data access:** each employee sees only their own path (Step 3.5), mirroring the right of access to one's own data.
- **Aggregate-only reporting:** KPI views show counts and percentages, never individual rows (Step 3.6, NB04).
- **Human in the loop:** an L&D reviewer approves course matches before they reach learners (Step 3.2), so recommendations are not purely automated decisions.

Documented, not built (roadmap):
- Formal data-minimisation audit of employee fields.
- Encryption at rest, consent management, right to erasure, retention policy.
- Real authentication and role-based access (the login in Step 3.5 is simulated).

Note: all data is synthetic. This shows a privacy-by-design pattern, not legal compliance.

### __Step 3.6: KPI Impact Rollup__

- Shows how the learning paths connect back to strategic goal G2 (ADR-0036, ADR-0037).
- **K2.2 (gap-closure rate, target 6 KSBs/year):**
  - Actual: gaps closed between last year's and this year's assessment.
  - Projected: open gaps each path addresses.
- **K2.1 (% certified in DS+ML fundamentals, target 60%):**
  - Proxy: share of focus-role employees meeting the requirement on all 3 primary K2.1 competencies (`kpi_competency_mapping`).
- Two projections, so the number is not oversold:
  - Optimistic: every gap with an approved course closes.
  - Conservative: only 1-level gaps close in one cycle; bigger gaps take longer.
- K1.1 and K1.2 are out of scope. They are client-level KPIs tracked outside competency data.
- Totals only, saved to `outputs/`. This is a projection, not a measured outcome (Logic Model, ADR-0021).  
- Logic Model (ADR-0021): the project builds up to Outputs (gaps, matches, paths) but does not simulate Outcomes or Impact, since that would need real follow-up data.

In [9]:
# Step 3.6: KPI Impact Rollup (aggregate only, ADR-0036, ADR-0037)
# Reads: data/raw/assessment_responses.csv, assessment_questions.csv, role_competency_requirements.csv
#        (in memory) employee_ksb_gaps, path_candidates, kpi_competency_mapping, kpis, employees, ksbs
# Saves: outputs/kpi_impact_rollup.csv

KSBS_PER_COMPETENCY = 3   # each competency gap cascades to its 3 KSBs (ADR-0028)
assessment_questions = pd.read_csv(os.path.join(RAW_DIR, 'assessment_questions.csv'))
assessment_responses = pd.read_csv(os.path.join(RAW_DIR, 'assessment_responses.csv'))
role_competency_requirements = pd.read_csv(os.path.join(RAW_DIR, 'role_competency_requirements.csv'))

# ---------- K2.2 actual: gaps closed between prior_year and current_year ----------
prof_q = assessment_questions[assessment_questions['question_type'] == 'proficiency'][['question_id', 'linked_competency_id']]
prof = (assessment_responses.merge(prof_q, on='question_id')
        .rename(columns={'linked_competency_id': 'competency_id', 'response_value': 'actual_level'})
        .merge(employees[['employee_id', 'role_id']], on='employee_id')
        .merge(role_competency_requirements, on=['role_id', 'competency_id']))
prof['gap'] = prof['required_level'] - prof['actual_level']
cycles = prof.pivot_table(index=['employee_id', 'competency_id'], columns='assessment_cycle', values='gap').reset_index()
cycles['closed'] = (cycles['prior_year'] > 0) & (cycles['current_year'] <= 0)

closed_per_emp = cycles.groupby('employee_id')['closed'].sum() * KSBS_PER_COMPETENCY
k22_actual = closed_per_emp.mean()

# ---------- K2.2 projected: open gaps each path addresses ----------
with_course = path_candidates[~path_candidates['is_content_gap']]
k22_proj_opt = with_course.groupby('employee_id').size().reindex(closed_per_emp.index, fill_value=0).mean()
k22_proj_cons = (with_course[with_course['gap_size'] == 1].groupby('employee_id').size()
                 .reindex(closed_per_emp.index, fill_value=0).mean())

# ---------- K2.1 proxy: meets requirement on all primary K2.1 competencies ----------
primary_comps = kpi_competency_mapping.loc[kpi_competency_mapping['contribution_weight'] == 'primary', 'competency_id'].tolist()
comp_gap = (employee_ksb_gaps.merge(ksbs[['ksb_id', 'competency_id']], on='ksb_id')
            .groupby(['employee_id', 'competency_id'])['gap_size'].first().reset_index())
comp_gap = comp_gap[comp_gap['competency_id'].isin(primary_comps)]
content_gap_comps = path_candidates.groupby(['employee_id', 'competency_id'])['is_content_gap'].any().rename('has_content_gap')
comp_gap = comp_gap.merge(content_gap_comps, on=['employee_id', 'competency_id'], how='left') # Set True/False type explicitly before filling blanks; avoids the pandas FutureWarning on silent downcasting.
comp_gap['has_content_gap'] = comp_gap['has_content_gap'].astype('boolean').fillna(False).astype(bool)

comp_gap['met_now'] = comp_gap['gap_size'] <= 0
comp_gap['met_opt'] = comp_gap['met_now'] | ~comp_gap['has_content_gap']
comp_gap['met_cons'] = comp_gap['met_now'] | ((comp_gap['gap_size'] == 1) & ~comp_gap['has_content_gap'])
k21 = comp_gap.groupby('employee_id')[['met_now', 'met_opt', 'met_cons']].all().mean() * 100

# ---------- Save aggregate rollup ----------
target = dict(zip(kpis['kpi_id'], kpis['target_value']))
rollup = pd.DataFrame([
    ['K2.2', 'Actual KSBs closed per employee (prior to current year)', round(k22_actual, 1), target['K2.2']],
    ['K2.2', 'Projected KSBs closed per employee (optimistic)', round(k22_proj_opt, 1), target['K2.2']],
    ['K2.2', 'Projected KSBs closed per employee (conservative)', round(k22_proj_cons, 1), target['K2.2']],
    ['K2.1', '% meeting all primary K2.1 competencies (now)', round(k21['met_now'], 1), target['K2.1']],
    ['K2.1', '% meeting all primary K2.1 competencies (optimistic)', round(k21['met_opt'], 1), target['K2.1']],
    ['K2.1', '% meeting all primary K2.1 competencies (conservative)', round(k21['met_cons'], 1), target['K2.1']],
], columns=['kpi_id', 'metric', 'value', 'target'])
rollup.to_csv(os.path.join(OUTPUT_DIR, 'kpi_impact_rollup.csv'), index=False)

print("Step 3.6 executed: kpi_impact_rollup.csv saved to outputs (aggregate only)")
print(f"Population: {closed_per_emp.shape[0]} focus-role employees | K2.1 primary competencies: {primary_comps}")
print()
print(rollup.to_string(index=False))

Step 3.6 executed: kpi_impact_rollup.csv saved to outputs (aggregate only)
Population: 55 focus-role employees | K2.1 primary competencies: ['COMP01', 'COMP02', 'COMP07']

kpi_id                                                  metric  value      target
  K2.2 Actual KSBs closed per employee (prior to current year)    3.2 6 KSBs/year
  K2.2         Projected KSBs closed per employee (optimistic)   12.1 6 KSBs/year
  K2.2       Projected KSBs closed per employee (conservative)    9.6 6 KSBs/year
  K2.1           % meeting all primary K2.1 competencies (now)    9.1         60%
  K2.1    % meeting all primary K2.1 competencies (optimistic)  100.0         60%
  K2.1  % meeting all primary K2.1 competencies (conservative)   70.9         60%


### __Step 3.6 Notes/Insights__

- K2.2 last year: 3.2 skills closed per person, about half the target of 6.
- K2.2 with paths: 9.6 to 12.1 skills per person. Too much for one year, so spread paths over 2 cycles.
- K2.1 now: only 9% meet all 3 core competencies.
- K2.1 with paths: 71% to 100%, above the 60% target.
- These are best-case projections. They assume people complete their paths.
- **Note:** Completing a course does not prove a gap is closed; that needs reassessment, manager feedback, on-the-job practice (70-20-10) and on-the-job evidence .

### __NB03 Summary and handoff to NB04__

What NB03 did:
- L&D review of 19 uncertain matches: 12 approved, 4 switched, 2 replaced, 1 content gap (KSB018).
- Built ordered learning paths for all 55 employees (687 rows, 5 to 21 steps each).
- Biggest gaps come first; Knowledge before Skill before Behavior.
- A shared course shows once; KSB018 shows as "coming soon" with manager or mentor support.
- Each employee sees only their own path (privacy by design).
- KPI rollup: paths could lift K2.1 from 9% to 71% or more, but only if learning transfers to the job.

Fixes made along the way:
- KPI fixes moved into NB01 code so they stick.
- Employee IDs now follow joining order.
- Re-run guards protect `courses.csv` and the review sheet.

Next, NB04 (visualization):
- Compare two contrasting employee paths (ID Lead vs Solution Architect).
- Build aggregate-only KPI dashboards from `kpi_impact_rollup.csv`.
- Show the content gap and the human review step visually.

### __End of NB03__